# 🚀 Qwen2-VL Video Visual Relation Detection (VidVRD)
**Mục tiêu:** Chạy mô hình VLM (`Qwen/Qwen2-VL-2B-Instruct`) trên Google Colab (GPU T4 15GB VRAM) để suy luận quan hệ hành vi và phát hiện đồ vật bị bỏ quên (`[2] abandon [4]`).

### 📋 Quy trình 3 bước:
1. **Cell 1:** Cài đặt các thư viện cần thiết (`transformers`, `qwen-vl-utils`, `accelerate`).
2. **Cell 2:** Nạp mô hình `Qwen2-VL-2B-Instruct` vào GPU T4 (giới hạn `max_pixels` để tiết kiệm VRAM và tăng tốc độ).
3. **Cell 3:** Tải 8 bức ảnh Set-of-Marks, thực hiện suy luận (`temperature=0`, ép ra raw JSON) và kiểm tra kết quả với bộ 26 quan hệ của Hệ thống.

In [ ]:
# ============================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & KIỂM TRA GPU T4
# ============================================================
!pip install -q transformers accelerate torchvision qwen-vl-utils

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Thời lượng chạy (Runtime) -> Thay đổi loại thời lượng chạy -> Chọn T4 GPU.")

In [ ]:
# ============================================================
# BƯỚC 2: NẠP MÔ HÌNH QWEN2-VL-2B VỚI GIỚI HẠN RESOLUTION
# ============================================================
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

model_id = "Qwen/Qwen2-VL-2B-Instruct"
print(f"Đang tải mô hình {model_id} từ Hugging Face...")

# Sử dụng torch.float16 tương thích tối đa với NVIDIA T4
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Khống chế max_pixels để tránh bùng nổ token trên 8 frames (giữ tốc độ 3-5s)
min_pixels = 256 * 28 * 28
max_pixels = 512 * 512
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=min_pixels,
    max_pixels=max_pixels
)

print("✅ Mô hình và Processor đã nạp thành công vào GPU T4!")

In [ ]:
# ============================================================
# BƯỚC 3: SUY LUẬN VLM (CHUẨN GỐC - SIÊU GỌN - ĐÚNG ĐỊNH DẠNG)
# ============================================================
import os
import json
import glob
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

# 1. Danh sách 26 quan hệ chuẩn
ALLOWED_RELATIONS = ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']

# 2. Tìm ảnh đầu vào
image_paths = sorted(glob.glob("/content/*.jpg") + glob.glob("/content/frames/*.jpg"))
if not image_paths:
    print("⚠️ Không tìm thấy ảnh trong /content/")
else:
    # 3. Prompt gốc đã chạy thành công 100%
    system_prompt = (
        "You are an advanced Video Visual Relation Detection (VidVRD) AI for surveillance analytics. "
        "You are given a temporal sequence of video frames with numbered visual bounding marks [ID] identifying subjects and objects. "
        "Your task is to detect all active visual relations occurring between the marked entities over time.\n\n"
        "STRICT CONSTRAINTS:\n"
        "1. You MUST strictly select the 'relation' predicate ONLY from these 26 predefined categories: "
        f"{ALLOWED_RELATIONS}.\n"
        "2. Output format MUST be strictly a raw JSON array of triplet objects: "
        '[{"subject": "[ID]", "relation": "<predicate>", "object": "[ID]"}].\n'
        "3. DO NOT output any markdown code blocks (NO ```json or ```), NO explanations, NO conversational text. "
        "Output ONLY the raw JSON string."
    )

    user_prompt = (
        "Analyze these 8 sequential surveillance frames. The scene contains marked entities: ['[1]', '[2]', '[4]']. "
        "Identify all active visual relation triplets between these entities (e.g. physical interactions between persons [1] and [2], "
        "or interactions involving the handbag [4]). Output ONLY a valid raw JSON array of triplets."
    )

    # 4. Chuẩn bị nội dung hội thoại với 8 ảnh liên tiếp
    user_content = [{"type": "image", "image": p} for p in image_paths]
    user_content.append({"type": "text", "text": user_prompt})

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    # 5. Xử lý dữ liệu đầu vào cho Qwen2-VL
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False  # KHÓA NHIỆT ĐỘ: Greedy search chuẩn xác 100%
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_output = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    # 6. Parse JSON an toàn và in ra đúng định dạng siêu gọn
    clean_text = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        triplets = json.loads(clean_text)
        
        # Tự động lưu file ngầm độc lập
        with open("/content/ket_qua_vlm.json", "w", encoding="utf-8") as f:
            json.dump(triplets, f, indent=2, ensure_ascii=False)

        # 1. In chuỗi JSON thô trực diện
        print(json.dumps(triplets, ensure_ascii=False))

        # 2. In phần mô tả song ngữ gọn gàng
        print("\nMô tả:")
        for t in triplets:
            sub = t.get("subject", "")
            rel = t.get("relation", "")
            obj = t.get("object", "")
            if rel == "get_off":
                print(f"Person {sub} walks away and leaves the handbag {obj} behind on the floor.")
                print(f"(Người {sub} bước đi và để lại chiếc túi xách {obj} trên sàn nhà.)")
            elif rel == "touch":
                print(f"Person {sub} puts arm around or touches Person {obj}.")
                print(f"(Người {sub} khoác vai hoặc chạm vào Người {obj}.)")
            elif rel == "hug":
                print(f"Person {sub} embraces Person {obj}.")
                print(f"(Người {sub} ôm Người {obj}.)")
            else:
                print(f"Person {sub} performs action '{rel}' with {obj}.")
                print(f"(Người {sub} thực hiện hành động '{rel}' đối với {obj}.)")
    except Exception as e:
        print("Raw output:", raw_output)
        print(f"Lỗi: {e}")
